In [ ]:
# pip install faiss-cpu

In [ ]:
import faiss

In [ ]:
faiss.__version__

In [ ]:
bio_text = """
        Your_Data
    """

In [ ]:
import requests
import json
import numpy as np

### Chunking and Overlap

In [ ]:
def chunk_text(text, chunk_size=10, overlap=2):   # Modify the 'chunk_size' and 'overlap' as required
    words = text.split()
    chunks = []
    start = 0
    
    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunk = ' '.join(words[start:end])
        chunks.append(chunk)
        start += chunk_size - overlap
    
    return chunks

In [ ]:
chunks = chunk_text(bio_text)

In [ ]:
chunks

### Initializing Embedding setup

In [ ]:
API_URL = "https://api.euron.one/api/v1/euri/embeddings"
API_KEY = "Your_EURI_API_Key"   # Replace with your actual API key
MODEL_NAME = "text-embedding-3-small"   # Replace with any other model of your choice/requirement

In [ ]:
header = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {API_KEY}"
}

### Creating chunks of the entire dataset

In [ ]:
all_embeddings = []

for _, chunk in enumerate(chunks):
    
    payload = {
        "model": MODEL_NAME,
        "input": chunk
    }

    response = requests.post(API_URL, headers=header, data=json.dumps(payload))
    data = response.json()
    embeddings = data['data'][0]['embedding']
    all_embeddings.append(embeddings)

In [ ]:
all_embeddings

In [ ]:
len(all_embeddings)

In [ ]:
len(chunks)

In [ ]:
type(all_embeddings)

In [ ]:
embeddings_array = np.array(all_embeddings, dtype='float32')

In [ ]:
embeddings_array

### Creating Index (Table) in FAISS VectorDB

In [ ]:
base_index = faiss.IndexFlatL2(1536)   # L2: Euclidean distance

### Adding Embeddings to FAISS VectorDB

In [ ]:
base_index.add(embeddings_array)

### Persisting physically

In [ ]:
faiss.write_index(base_index, "faiss_index.faiss")

### Testing

In [ ]:
query_test = "tell me about kumar early life"

In [ ]:
def embedding_text(text):
    
    payload = {
        "model": MODEL_NAME,
        "input": text
    }

    response = requests.post(API_URL, headers=header, data=json.dumps(payload))
    data = response.json()
    embeddings = data['data'][0]['embedding']
    emb = np.array(embeddings, dtype="float32").reshape(1,-1)

    return emb

In [ ]:
query_test_emb = embedding_text(query_test)

In [ ]:
query_test_emb

### Searching in FAISS VectorDB

In [ ]:
base_index.search(query_test_emb, 3)

In [ ]:
chunks[4]

In [ ]:
chunks[8]

In [ ]:
chunks[6]